In [48]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.utils import resample

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv
/kaggle/input/datasets/amanalisiddiqui/fraud-detection-dataset/AIML Dataset.csv


In [49]:
df_trans = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
df_id = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
# preserve all transactions
df = df_trans.merge(df_id, on='TransactionID', how='left')

**DATASET**

In [50]:
print(df.head())

   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ...                id_31  id_32  \
0    NaN  150.0    discover  142.0  ...                  NaN    NaN   
1  404.0  150.0  mastercard  102.0  ...                  NaN    NaN   
2  490.0  150.0        visa  166.0  ...                  NaN    NaN   
3  567.0  150.0  mastercard  117.0  ...                  NaN    NaN   
4  514.0  150.0  mastercard  102.0  ...  samsung browser 6.2   32.0   

       id_33           id_34  id_35 id_36 id_37  id_38  DeviceType  \
0        NaN             NaN    NaN   Na

In [51]:
print(df.columns.tolist)
print(f"Size of transaction dataset: {df_trans.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Size of identity dataset: {df_id.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

<bound method IndexOpsMixin.tolist of Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5',
       ...
       'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38',
       'DeviceType', 'DeviceInfo'],
      dtype='object', length=434)>
Size of transaction dataset: 2062.07 MB
Size of identity dataset: 143.14 MB


**EDA**

In [52]:
print(df['isFraud'].value_counts(normalize=True))

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [53]:
dtype_df = pd.DataFrame(df.dtypes.reset_index())
dtype_df.columns = ['Column', 'Data Type']
print(dtype_df)

             Column Data Type
0     TransactionID     int64
1           isFraud     int64
2     TransactionDT     int64
3    TransactionAmt   float64
4         ProductCD    object
..              ...       ...
429           id_36    object
430           id_37    object
431           id_38    object
432      DeviceType    object
433      DeviceInfo    object

[434 rows x 2 columns]


In [54]:
print(df.dtypes.value_counts())

float64    399
object      31
int64        4
Name: count, dtype: int64


In [55]:
results = {}
def find_nans(df, target='isFraud'):
    
    missing_pct = df.isna().mean() * 100
    high_missing = missing_pct[missing_pct > 50].index.tolist()
    results['high_missing_cols'] = high_missing
    print(f"Columns with >50% NaN: {high_missing}")
    
    placeholder_values = ['', 'NULL', 'null', 'None', 'NA', 'N/A', '-1', '-999']
    
    for col in df.select_dtypes(include=['object']).columns:
        for placeholder in placeholder_values:
            if placeholder in df[col].values:
                pct = (df[col] == placeholder).mean() * 100
                if pct > 0:
                    print(f"Column '{col}' has {pct:.1f}% '{placeholder}' values")
    
    return

eda(df)

Columns with >50% NaN: ['dist1', 'dist2', 'R_emaildomain', 'D5', 'D6', 'D7', 'D8', 'D9', 'D12', 'D13', 'D14', 'M5', 'M7', 'M8', 'M9', 'V138', 'V139', 'V140', 'V141', 'V142', 'V143', 'V144', 'V145', 'V146', 'V147', 'V148', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V161', 'V162', 'V163', 'V164', 'V165', 'V166', 'V167', 'V168', 'V169', 'V170', 'V171', 'V172', 'V173', 'V174', 'V175', 'V176', 'V177', 'V178', 'V179', 'V180', 'V181', 'V182', 'V183', 'V184', 'V185', 'V186', 'V187', 'V188', 'V189', 'V190', 'V191', 'V192', 'V193', 'V194', 'V195', 'V196', 'V197', 'V198', 'V199', 'V200', 'V201', 'V202', 'V203', 'V204', 'V205', 'V206', 'V207', 'V208', 'V209', 'V210', 'V211', 'V212', 'V213', 'V214', 'V215', 'V216', 'V217', 'V218', 'V219', 'V220', 'V221', 'V222', 'V223', 'V224', 'V225', 'V226', 'V227', 'V228', 'V229', 'V230', 'V231', 'V232', 'V233', 'V234', 'V235', 'V236', 'V237', 'V238', 'V239', 'V240', 'V241', 'V242', 'V243', 'V244', 'V245', 'V

In [56]:
cat_cols = df.select_dtypes(include=['object']).columns
fraud_rates = {}  

# proportion of ones
global_fraud_rate = df['isFraud'].mean()

for col in cat_cols:
    fraud_rate = df.groupby(col)['isFraud'].mean().sort_values(ascending=False)
    fraud_rates[col] = fraud_rate
    
    # columns where max fraud rate is bigger than two times
    # the global average
    if fraud_rate.max() > global_fraud_rate * 2:
        print(f"\nFraud rate by {col}:")
        print(fraud_rate.head(5))


Fraud rate by ProductCD:
ProductCD
C    0.116873
S    0.058996
H    0.047662
R    0.037826
W    0.020399
Name: isFraud, dtype: float64

Fraud rate by card4:
card4
discover            0.077282
visa                0.034756
mastercard          0.034331
american express    0.028698
Name: isFraud, dtype: float64

Fraud rate by P_emaildomain:
P_emaildomain
protonmail.com    0.407895
mail.com          0.189624
outlook.es        0.130137
aim.com           0.126984
outlook.com       0.094584
Name: isFraud, dtype: float64

Fraud rate by R_emaildomain:
R_emaildomain
protonmail.com    0.951220
mail.com          0.377049
netzero.net       0.222222
outlook.com       0.165138
outlook.es        0.131640
Name: isFraud, dtype: float64

Fraud rate by M4:
M4
M2    0.113739
M0    0.036649
M1    0.027051
Name: isFraud, dtype: float64

Fraud rate by id_12:
id_12
NotFound    0.081683
Found       0.059836
Name: isFraud, dtype: float64

Fraud rate by id_15:
id_15
Found      0.105097
Unknown    0.091885
New    

In [ ]:
from scipy.stats import ks_2samp

numeric_cols = df.select_dtypes(include=[np.number]).columns
ks_results = []

#1% of fraud samples OR 30, whichever is larger
min_fraud_samples = max(30, int(fraud_count * 0.01))

for col in numeric_cols:
    fraud_vals = df[df['isFraud'] == 1][col].dropna()
    legit_vals = df[df['isFraud'] == 0][col].dropna()

    """
    Small sample sizes are excluded because the shape of the 
    distributions for small samples.

    The empirical distribution from a small sample is unlikely to 
    represent the true population distribution, 
    leading to unreliable and non-reproducible KS values.

    Therefore, the KS statistic is unstable in that case.
    """
    
    if len(fraud_vals) >= min_fraud_samples and len(legit_vals) >= min_fraud_samples:
        ks_stat = ks_2samp(fraud_vals, legit_vals).statistic
        ks_stat = ks_2samp(fraud_vals, legit_vals).statistic
        ks_results.append((col, ks_stat, len(fraud_vals)))  

ks_df = pd.DataFrame(ks_results, columns=['feature', 'KS', 'fraud_samples'])
top_ks = ks_df.sort_values('KS', ascending=False).head(10)

"""
columns with higher ks statistic show more clear separation
between data points from one class and data points from the other
"""
print(f"\nTop 10 by KS statistic:\n{top_ks}")

**TIME-BASED SPLIT**

In [58]:
df = df.sort_values('TransactionDT')
X = df.drop('isFraud', axis=1)
y = df['isFraud']

"""
For big datasets, n-splits = 10 is a good trafeoff between the size 
of each training set and the number of prediction sets
"""
tscv = TimeSeriesSplit(n_splits=10)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    print(f"Fold {fold}: Train rows = {len(X_train)}, Test rows = {len(X_test)}, Fraud in test = {y_test.sum()}")

df = df.sort_values('TransactionDT')
X = df.drop('isFraud', axis=1)
y = df['isFraud']

tscv = TimeSeriesSplit(n_splits=10)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
print(f"\nTotal test rows across all folds: {sum([len(X.iloc[test_idx]) for _, (_, test_idx) in enumerate(tscv.split(X))])}")
print(f"Average test rows per fold: {sum([len(X.iloc[test_idx]) for _, (_, test_idx) in enumerate(tscv.split(X))]) / tscv.n_splits:.0f}")

Fold 1: Train rows = 53690, Test rows = 53685, Fraud in test = 1234
Fold 2: Train rows = 107375, Test rows = 53685, Fraud in test = 1727
Fold 3: Train rows = 161060, Test rows = 53685, Fraud in test = 2145
Fold 4: Train rows = 214745, Test rows = 53685, Fraud in test = 2376
Fold 5: Train rows = 268430, Test rows = 53685, Fraud in test = 2020
Fold 6: Train rows = 322115, Test rows = 53685, Fraud in test = 1856
Fold 7: Train rows = 375800, Test rows = 53685, Fraud in test = 2271
Fold 8: Train rows = 429485, Test rows = 53685, Fraud in test = 1940
Fold 9: Train rows = 483170, Test rows = 53685, Fraud in test = 1636
Fold 10: Train rows = 536855, Test rows = 53685, Fraud in test = 2006

Total test rows across all folds: 536850
Average test rows per fold: 53685


**SCALING**

In [59]:
"""
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
"""

'\nscaler = RobustScaler()\nX_train_scaled = scaler.fit_transform(X_train)\nX_test_scaled = scaler.transform(X_test)\n'